In [ ]:
# sql connections
import mysql.connector

# Function to Insert data into Table
def insertDataIntoSqlTable(df,tableName):
 # Step 1: Establish connection
    connection = mysql.connector.connect(
     host="localhost",
     user="root",
     password="sql@1234",
     database="logistics_dataset"
     )

 #create a cursor
    cursor = connection.cursor()

 # Execute Insert query
    if tableName == "warehouses":
         query = "INSERT INTO warehouses (warehouse_id, city,state,capacity) VALUES (%s, %s,%s,%s) ON DUPLICATE KEY UPDATE city = VALUES(city),state = VALUES(state),capacity = VALUES(capacity)"

    elif tableName == "shipments" :
        query = """INSERT INTO shipments (shipment_id, order_date,origin,destination,weight,courier_id,status,delivery_date) VALUES (%s, %s,%s,%s,%s, %s,%s,%s) 
        ON DUPLICATE KEY UPDATE order_date = VALUES(order_date),origin = VALUES(origin),
        destination = VALUES(destination),weight = VALUES(weight),
        courier_id = VALUES(courier_id),status = VALUES(status),delivery_date = VALUES(delivery_date)"""
    elif tableName == "costs":
        query = """INSERT INTO costs (shipment_id, fuel_cost,labor_cost,misc_cost) VALUES (%s, %s,%s,%s)
        ON DUPLICATE KEY UPDATE fuel_cost = VALUES(fuel_cost),labor_cost = VALUES(labor_cost),
        misc_cost = VALUES(misc_cost)"""
    elif tableName == "routes":
        query = """INSERT INTO routes (route_id, origin,destination,distance_km,avg_time_hours) VALUES (%s, %s,%s,%s,%s)
        ON DUPLICATE KEY UPDATE origin = VALUES(origin),destination = VALUES(destination),
        distance_km = VALUES(distance_km),avg_time_hours = VALUES(avg_time_hours)"""
    elif tableName == "courier_staff":
        query = """INSERT INTO courier_staff (courier_id, name,rating,vehicle_type) VALUES (%s,%s,%s,%s)
         ON DUPLICATE KEY UPDATE name = VALUES(name),rating = VALUES(rating),
         vehicle_type = VALUES(vehicle_type)"""
    elif tableName =="shipment_tracking":
        query = """INSERT INTO shipment_tracking (tracking_id, shipment_id,status,timestamp) VALUES (%s, %s,%s,%s)
         ON DUPLICATE KEY UPDATE shipment_id = VALUES(shipment_id),status = VALUES(status),
         timestamp = VALUES(timestamp)"""

 
    values = list(df.itertuples(index=False, name=None))
 
    cursor.executemany(query, values)

    connection.commit()

    print(cursor.rowcount, f"rows inserted into {tableName}")
 # Step 6: Close connection
    cursor.close()
    connection.close()

 
 

'     print(row) '

In [ ]:
import json
import pandas as pd
import numpy as np
#step 1 : Load json files and csv files Into DataFrames
# first load MasterTable data then referneced table data 
#Master Tables  = warehouses,routes,courier_staff
#Tables That has refrenced data = costs,shipments,shipment_tracking

try:
 
#1 : Load warehouse data and insert into Sqltable
 with open(r"../data/warehouses.json",'r') as file:
   data = json.load(file)
# Create DataFrame
 df = pd.DataFrame.from_dict(data)
 # drop duplicates from the dataset
 df = df.drop_duplicates(subset=["warehouse_id"])
 insertDataIntoSqlTable(df,"warehouses")

#2: Load courier_staff data and insert into sql table
 df = pd.read_csv(r'..\data\courier_staff.csv')
 df = df.drop_duplicates(subset=["courier_id"])
 insertDataIntoSqlTable(df,"courier_staff")

#3 : Load routes data and insert into sql table
 df = pd.read_csv(r'..\data\routes.csv')
 df = df.drop_duplicates(subset=["route_id"])
 insertDataIntoSqlTable(df,"routes")

#4: load shipments data and insert into sql table
 with open(r"../data/shipments.json",'r') as file:
  data = json.load(file)
 #Create DataFrame
 df = pd.DataFrame.from_dict(data)
 # drop duplicates from the dataset
 df = df.drop_duplicates(subset=["shipment_id"])
 df = df.replace({np.nan: None})
 insertDataIntoSqlTable(df,"shipments")

#5: load shipment_tracking data and insert into sql table
 df = pd.read_csv(r'..\data\shipment_tracking.csv')
 df = df.drop_duplicates(subset=["tracking_id"])
 insertDataIntoSqlTable(df,"shipment_tracking")

#6 load costs data and insert into sqltable
 df = pd.read_csv(r'..\data\costs.csv')
 df = df.drop_duplicates(subset=["shipment_id"])
 insertDataIntoSqlTable(df,"costs")


except json.JSONDecodeError as e:
  print("Invaliod JSON",e)
except FileNotFoundError:
  print("File not found")

    courier_id                name  rating vehicle_type
0     7bc57152        Justin Flynn     4.8          Van
1     87cafee3       Sherry Conrad     3.2        Truck
2     53cc593b          Joanna Liu     4.0          Car
3     c482832b        Samantha Lee     4.9          Car
4     a217d894       Carrie Santos     3.9          Van
..         ...                 ...     ...          ...
995   1bafc418     Andrea Robinson     3.9          Car
996   55f3ed49  Benjamin Middleton     3.8          Van
997   60912cf1    Michele Mcknight     4.5          Car
998   1199b2f9     Johnny Johnston     4.7        Truck
999   78434baa       Melissa Weber     3.0          Van

[1000 rows x 4 columns]
0 rows inserted into warehouses
0 rows inserted into courier_staff
0 rows inserted into routes
69999 rows inserted into shipments
209570 rows inserted into shipment_tracking
69999 rows inserted into costs
